# Chapter 2: AAV6 Frequency changes indicates tissue and process specific in vivo selection 
# Create AAV6-ML figures for Chapter 2 for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 21/04/2026
Finished: 21/04/2026

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2
from scipy.stats import pearsonr, spearmanr, linregress
from matplotlib.patches import Patch


## 2. Preperation 
### 2.1. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for figures
figures_dir = general_dir/'figures'/'2_RPM'
os.makedirs(figures_dir, exist_ok=True)

### 2.2. Define helper functions


In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
# sort file names from df_long
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

In [ ]:
# compare and corr two tables
def compare_two_tables(
    df1,
    df2,
    name1="sample_1",
    name2="sample_2",
    compare_cols=None,
    key_col="AA_sequence",
    log_scale_cols=None,
    add_diagonal=True,
    alpha=0.25,
    point_size=8,
    figsize_per_panel=(5.5, 5),
    save=False,
    output_path=None,
    file_name=None,
    show=True
):
    """
    Compare two tables across one or multiple numeric columns by merging on key_col.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Tables to compare
    name1, name2 : str
        Names shown in axis labels
    compare_cols : list of str
        Numeric columns to compare, e.g. ["Log2_enrichment", "RPM"]
    key_col : str
        Merge key, default = "AA_sequence"
    log_scale_cols : list of str
        Columns that should be plotted on log-log axes, e.g. ["RPM"]
    add_diagonal : bool
        Whether to add y=x diagonal
    alpha : float
        Point transparency
    point_size : int
        Scatter point size
    figsize_per_panel : tuple
        Width, height per subplot
    save : bool
        Whether to save figure
    output_path : str or Path
        Save directory
    file_name : str
        Output filename
    show : bool
        Whether to show plot

    Returns
    -------
    merged : pd.DataFrame
        Merged comparison table
    corr_df : pd.DataFrame
        Correlation summary table
    """

    if compare_cols is None:
        compare_cols = ["Log2_enrichment", "RPM"]

    if log_scale_cols is None:
        log_scale_cols = []

    # keep only needed columns
    cols1 = [key_col] + compare_cols
    cols2 = [key_col] + compare_cols

    a = df1[cols1].copy()
    b = df2[cols2].copy()

    # rename value columns
    a = a.rename(columns={col: f"{col}_{name1}" for col in compare_cols})
    b = b.rename(columns={col: f"{col}_{name2}" for col in compare_cols})

    # merge
    merged = a.merge(b, on=key_col, how="inner")

    # correlation results
    results = []

    # figure layout
    n = len(compare_cols)
    ncols = min(2, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows)
    )

    # make axes iterable
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).reshape(-1)

    for i, col in enumerate(compare_cols):
        ax = axes[i]

        xcol = f"{col}_{name1}"
        ycol = f"{col}_{name2}"

        sub = merged[[xcol, ycol]].dropna().copy()

        # correlations
        if len(sub) > 1:
            pearson_r, pearson_p = pearsonr(sub[xcol], sub[ycol])
            spearman_r, spearman_p = spearmanr(sub[xcol], sub[ycol])
        else:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_r, spearman_p = np.nan, np.nan

        results.append({
            "column": col,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "n_points": len(sub)
        })

        # scatter
        sns.scatterplot(
            data=sub,
            x=xcol,
            y=ycol,
            s=point_size,
            alpha=alpha,
            ax=ax
        )

        # log scaling if requested
        if col in log_scale_cols:
            sub_pos = sub[(sub[xcol] > 0) & (sub[ycol] > 0)].copy()
            ax.clear()
            sns.scatterplot(
                data=sub_pos,
                x=xcol,
                y=ycol,
                s=point_size,
                alpha=alpha,
                ax=ax
            )
            ax.set_xscale("log")
            ax.set_yscale("log")
            plot_data = sub_pos
        else:
            plot_data = sub

        # diagonal
        if add_diagonal and len(plot_data) > 0:
            lims = [
                min(plot_data[xcol].min(), plot_data[ycol].min()),
                max(plot_data[xcol].max(), plot_data[ycol].max())
            ]
            ax.plot(lims, lims, "--", color="black", linewidth=1.2, alpha=0.6)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

        ax.set_xlabel(f"{col} ({name1})")
        ax.set_ylabel(f"{col} ({name2})")
        ax.set_title(col)

        ax.text(
            0.05, 0.95,
            f"Pearson r = {pearson_r:.4f}\nSpearman ρ = {spearman_r:.4f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        sns.despine(ax=ax)

    # remove unused axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        import os
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            col_string = "_".join(compare_cols)
            file_name = f"compare_{name1}_vs_{name2}_{col_string}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    corr_df = pd.DataFrame(results)

    return merged, corr_df

## 3. Script Functions

### 3.1. Function to create Histogram of sample RPM vs input RPM

In [ ]:
# Function to create Histogram of sample RPM vs input RPM
def distribution_histogram(
    table,
    tissue,
    extraction,
    replicate="biological",
    legend=True,
    title=True,
    save=False,
    output_path=None,
    file_name=None,
    figsize=(7.5, 5),
    input_alpha=1,
    sample_alpha=0.30
):
    """
    Plot RPM distribution for one tissue/extraction condition.

    Parameters
    ----------
    table : pd.DataFrame
    tissue : str
    extraction : str
    replicate : str
        'biological' or 'technical'
    legend : bool
    title : bool
    save : bool
    output_path : str or Path
    file_name : str
    figsize : tuple
        Figure size, e.g. (7.5, 5)
    input_alpha : float
        Transparency of input library histogram
    sample_alpha : float
        Transparency of sample histograms
    """

    # -----------------------------
    # Select sample data
    # -----------------------------
    cols = ["AA_sequence", "Mouse_ID", "Sample", "RPM", "Pseudo", "Replicate", "Tissue", "Extraction_type"]
    df = table.loc[
        (table["Replicate"] == replicate) &
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction),
        cols
    ].copy()

    # biological plots grouped by Mouse_ID, technical plots by Sample
    
    group_col, legend_title = 2*["Mouse_ID"]


    # -----------------------------
    # Prepare input library
    # -----------------------------
    input_cols = ["AA_sequence", "RPM"]
    input_library = dict_library[tissue].loc[:, input_cols].copy()
    input_library[group_col] = "input_library"

    df_all = pd.concat([input_library, df], axis=0, ignore_index=True)

    if df_all.empty:
        raise ValueError("No valid RPM values found for the selected filters.")

    # -----------------------------
    # Bin edges
    # -----------------------------
    x_min = table['RPM'].min()
    x_max = table['RPM'].max()
    
    bin_edges = np.logspace(np.log10(x_min), np.log10(x_max), 100)

    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=figsize)

    # -----------------------------
    # Colors
    # -----------------------------
    mouse_palette = {
        "f1": "#9ECAE1",
        "f2": "#6BAED6",
        "f3": "#3182BD",
        "m1": "#D4B9DA",
        "m2": "#C994C7",
        "m3": "#88419D"
    }

    input_color = "#BDBDBD"

    # -----------------------------
    # Plot input library first
    # -----------------------------
    ax.hist(
        input_library["RPM"],
        bins=bin_edges,
        alpha=input_alpha,
        color=input_color,
        edgecolor="none",
        zorder=1
    )

    # -----------------------------
    # Plot samples
    # -----------------------------

    ordered_groups = [x for x in MOUSE_ID]

    for grp in ordered_groups:
        sub = df.loc[df[group_col] == grp, "RPM"]
        color = mouse_palette.get(grp)

        ax.hist(
            sub,
            bins=bin_edges,
            alpha=sample_alpha,
            color=color,
            edgecolor="none",
            zorder=2
        )

    # -----------------------------
    # Axes
    # -----------------------------
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(x_min, x_max)

    ax.set_xlabel(f"Variant RPM in {extraction} {tissue}")
    ax.set_ylabel("Number of AA sequences")

    if title:
        ax.set_title(f"{extraction} {tissue}\n RPM distribution across {replicate} replicates")

    # -----------------------------
    # Legend
    # -----------------------------
    if legend:
        handles = [
            Patch(facecolor=input_color, edgecolor="none", alpha=input_alpha, label="input_library")
        ]

        for grp in ordered_groups:
            color = mouse_palette.get(grp)
            handles.append(
                Patch(facecolor=color, edgecolor="none", alpha=sample_alpha, label=str(grp))
            )

        ax.legend(
            handles=handles,
            title=legend_title,
            frameon=False,
            ncol=2,
            loc="upper right"
        )

    sns.despine()
    plt.tight_layout()

    # -----------------------------
    # Save
    # -----------------------------
    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"histogram_rpm_{extraction}_{tissue}_with_pseudo.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

### 3.2. Distribution ecdf for RPM and Log2_enrichment

In [ ]:
def distribution_ecdf(
    table,
    tissue,
    extraction,
    column,
    replicate,
    measure = 'median',
    no_pseudo = False,
    legend=True,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):

    """
    Plot empirical cumulative distribution functions (ECDFs) for one tissue and
    extraction condition, comparing the tissue-specific input library with either
    biological or technical replicate samples.

    The function overlays:
    - the tissue-specific input library
    - all selected sample distributions for the chosen replicate type

    and optionally adds reference lines for:
    - zero enrichment (for Log2-based columns)
    - the mean or median of the selected sample values

    Parameters
    ----------
    table : pd.DataFrame
        Long-format table containing per-variant sample information. Expected to
        include at least the columns:
        ['AA_sequence', 'Sample', 'Mouse_ID', 'Tissue', 'Extraction_type',
         'Replicate', column, 'Pseudo'].
    tissue : str
        Tissue to analyze, e.g. 'liver' or 'heart'.
    extraction : str
        Extraction type to analyze, e.g. 'gDNA' or 'cDNA'.
    column : str
        Numeric column to visualize as ECDF, e.g. 'RPM', 'Log2_enrichment',
        or another Log2-based metric.
    replicate : str
        Type of replicate grouping to use:
        - 'biological': curves are grouped by Mouse_ID
        - 'technical': curves are grouped by Sample
    measure : str, default='median'
        Summary statistic used for the vertical reference line across the selected
        sample data. Allowed values:
        - 'mean'
        - 'median'
    no_pseudo : bool, default=False
        If True, keep only rows with Pseudo == 0 in the selected sample data.
        For column='RPM', pseudo variants are removed regardless of this setting.
    legend : bool, default=True
        Whether to display the legend.
    title : bool, default=True
        Whether to display a plot title.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Output directory for saving the figure. Required if save=True.
    file_name : str, optional
        Output file name. If None, a default file name is generated.

    Returns
    -------
    None
        Displays the ECDF plot and optionally saves it to disk.

    Notes
    -----
    - The input library is taken from `dict_library[tissue]`.
    - For biological replicates, the legend groups curves by Mouse_ID.
    - For technical replicates, the legend groups curves by Sample.
    - For Log2-based columns, a vertical dashed line at x=0 is added to indicate
      no enrichment.
    - The mean/median reference line is calculated from the filtered sample data
      (`df_sample`), not from the concatenated input + sample table.
    - For RPM plots, the x-axis is shown on a log scale.
    """
    # -----------------------------
    # select type of replicates
    # -----------------------------
    if replicate == "biological":
        legend_name = "Data"
    elif replicate == "technical":
        legend_name = "Sample"
    else:
        raise ValueError("Replicate must be 'biological' or 'technical'")

    # -----------------------------
    # Prepare input library
    # -----------------------------
    input_library = dict_library[tissue].copy()
    input_library[legend_name] = "input_library"

    # -----------------------------
    # Prepare selected sample data
    # -----------------------------
    df_sample = table[
        (table["Replicate"] == replicate) &
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][["AA_sequence", "Sample", "Mouse_ID", f'{column}', "Pseudo"]].copy()
    
    # usage of pseudo variants or not
    if no_pseudo:
        df_sample = df_sample[df_sample['Pseudo'] == 0]
        
    if replicate == "biological":
        df_sample = df_sample.rename(columns={"Mouse_ID": "Data"})

    if column == 'RPM':
        df_sample = df_sample[df_sample['Pseudo'] == 0]

    # -----------------------------
    # Concat input + selected samples
    # -----------------------------
    df = pd.concat([input_library, df_sample], axis=0, ignore_index=True)

    # keep valid values only
    df = df[df[column].notna()].copy()

    # define x-range from filtered data
    x_min = df[column].min()
    x_max = df[column].max()

    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    # -----------------------------
    # Define colors same as histogram
    # -----------------------------
    color_palette = {
        "input_library": "#BDBDBD",  # grey
        "f1": "#9ECAE1",             # blue shades
        "f2": "#6BAED6",
        "f3": "#3182BD",
        "m1": "#D4B9DA",             # violet shades
        "m2": "#C994C7",
        "m3": "#88419D"
    }

    # -----------------------------
    # Define plotting order
    # -----------------------------
    present_samples = [x for x in df[legend_name].dropna().unique() if x != "input_library"]

    ordered_samples = [x for x in MOUSE_ID if x in present_samples] + \
                      [x for x in present_samples if x not in MOUSE_ID]

    sample_order = ["input_library"] + ordered_samples

    # -----------------------------
    # Plot ECDFs
    # -----------------------------
    for data_name in sample_order:
        sub = df[df[legend_name] == data_name].copy()
        if sub.empty:
            continue

        color = color_palette.get(data_name, "#4C72B0")

        # input in background, full opacity
        if data_name == "input_library":
            alpha_val = 1.0
            lw = 2.2
            z = 1
        else:
            alpha_val = 0.5
            lw = 2.0
            z = 2

        sns.ecdfplot(
            data=sub,
            x=column,
            ax=ax,
            color=color,
            alpha=alpha_val,
            linewidth=lw,
            label=data_name,
            zorder=z
        )

    # -----------------------------
    # Reference lines
    # -----------------------------

    if measure == 'mean':
        val= df_sample[column].mean()
    if measure == 'median':
        val= df_sample[column].median()

    # for RPM: only mean line
    # for log2_enrichment: zero line + mean/median line
    line_handles = []

    if column.startswith("Log2"):
        line_zero = ax.axvline(
            0,
            linestyle="--",
            linewidth=1.5,
            color="red",
            alpha=0.9,
            label="No enrichment = 0"
        )
        line_handles.append(line_zero)

    line_mean = ax.axvline(
        val,
        linestyle="--",
        linewidth=1.5,
        color="black",
        alpha=0.9,
        label=f"Mean = {val:.2e}" if measure == "mean" else f"Median = {val:.2f}"
    )

    line_handles.append(line_mean)

    # -----------------------------
    # Axes
    # -----------------------------
    ax.set_xlim(x_min, x_max)

    if column == "RPM":
        ax.set_xscale("log")
        ax.set_xlabel("Variant RPM")
        ylab = "Cumulative fraction of AA sequences"
        title_text = f"{extraction} {tissue}\nCumulative distribution RPMs"
    else:
        ax.set_xlabel(f"Variant {column}")
        ylab = "Cumulative fraction of AA sequences"
        title_text = f"{extraction} {tissue}\nCumulative distribution of {column}"

    ax.set_ylabel(ylab)

    if title:
        ax.set_title(title_text)

    # -----------------------------
    # Legend
    # -----------------------------
    if legend:
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))  # remove duplicates, preserve last occurrence
        ax.legend(
            unique.values(),
            unique.keys(),
            title= legend_name,
            loc="lower right",
            frameon=False,
            ncol=2
        )

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"ECDF_{column}_{extraction}_{tissue}_{replicate}.png"
        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

### 3.3. Venn2 Diagram of top RPM between sample and input

In [ ]:
def venn2_ntop_AA(
    table,
    tissue,
    extraction,
    n_top = 10000,
    figsize=(7, 4),
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
   
    """
    Venn diagram of  overlap between the top-N AA variants 
    in the pooled sample and the corresponding tissue-specific input library.

    Parameters
    ----------
    table : pd.DataFrame
        Pooled table containing at least the columns:
        ['AA_sequence', 'Tissue', 'Extraction_type', 'RPM'].
    tissue : str
        Tissue to filter for, e.g. 'liver' or 'heart'.
    extraction : str
        Extraction type to filter for, e.g. 'gDNA' or 'cDNA'.
    n_top : int, default = 10000
        Number of top variants (ranked by RPM) to include in the comparison.
    figsize : tuple, default=(7, 4)
    title : bool, default=True
    save : bool, default=False
    output_path : str or Path, optional

    Notes
    -----
    - The sample variants are taken from the pooled table.
    - The reference variants are taken from the tissue-specific input library
      stored in `dict_library[tissue]`.
    - Ranking is based on RPM in descending order.
    """
    
    # -----------------------------
    # Select sample I want to compare
    # -----------------------------
    sample = (
        table.loc[
            (table["Tissue"] == tissue) &
            (table["Extraction_type"] == extraction),
            ["AA_sequence", "RPM"]
        ]
        .sort_values("RPM", ascending=False)
        .head(n_top)
        .copy()
    )

    # -----------------------------
    # Select corresponding library
    # -----------------------------
    input_library = (
        dict_library[tissue]
            .loc[:, ["AA_sequence", "RPM"]]
            .sort_values("RPM", ascending=False)
            .head(n_top)
            .copy()
    )
    
    # --------------------------------------------------
    # Create figure and Venn diagram
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    v = venn2(
        [set(input_library["AA_sequence"]), set(sample["AA_sequence"])],
        set_labels=("Input library", f"{extraction} {tissue} pooled"),
        ax=ax
    )

    # --------------------------------------------------
    # Adjust font size of overlap counts and set labels
    # --------------------------------------------------
    if v.subset_labels is not None:
        for text in v.subset_labels:
            if text is not None:
                text.set_fontsize(18)
                text.set_fontweight("bold")

    if v.set_labels is not None:
        for text in v.set_labels:
            if text is not None:
                text.set_fontsize(14)

    # --------------------------------------------------
    # Optional title
    # --------------------------------------------------
    if title:
        ax.set_title(f"{extraction} {tissue}\nOverlap of top {n_top} RPM AA variants")

    plt.tight_layout()

    # --------------------------------------------------
    # Optional saving
    # --------------------------------------------------
    if save:
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"Venn2_{extraction}_{tissue}_RPM_top_{n_top}.png" 
        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

### 3.4. Rank shift RPM input library -> gDNA and cDNA

In [ ]:
def rank_shift_plot(
    table,
    tissue,
    extraction,
    n_sample=0,
    show_pseudo=True,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    """
    Plot the rank shift of AA variants between the tissue-specific input library
    and the mean biological sample RPM across all Mouse_IDs.

    The function:
    1. extracts biological replicate RPM values for one tissue/extraction condition,
    2. pivots them into wide format (one column per Mouse_ID),
    3. merges tissue-specific input library RPM values,
    4. calculates variant ranks in the input library and in each biological replicate,
    5. computes the mean rank across biological replicates,
    6. visualizes the shift between input rank and mean sample rank.

    Parameters
    ----------
    table : pd.DataFrame
        Long-format table containing at least:
        ['AA_sequence', 'Mouse_ID', 'Tissue', 'Extraction_type', 'Replicate', 'RPM'].
    tissue : str
        Tissue to analyze, e.g. 'liver' or 'heart'.
    extraction : str
        Extraction type to analyze, e.g. 'gDNA' or 'cDNA'.
    n_sample : int, default=0
        Variants present in >= n_sample are treated as normal.
        Variants present in < n_sample are treated as pseudo-supported.
    show_pseudo : bool, default=True
        Whether to plot pseudo-supported variants separately in red.
    title : bool, default=True
        Whether to display a title.
    show_table : bool, default=False
        Whether to display the top rows of the filtered non-pseudo table.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Output directory.
    file_name : str, optional
        Custom output file name. If None, a default name is generated.

    Returns
    -------
    normal : pd.DataFrame
        Table of variants with Pseudo >= n_sample.
    pseudo : pd.DataFrame
        Table of variants with Pseudo < n_sample.
        Empty if show_pseudo=False.
    df_merged : pd.DataFrame
        Full processed table used for plotting.

    Notes
    -----
    - Input library RPM values are taken from `dict_library[tissue]`.
    - n_sample information is taken from `df_pooled` for the selected
      tissue/extraction combination.
    - Ranking is based on descending RPM (highest RPM =  rank number 1).
    """

    # --------------------------------------------------
    # Select tissue-specific input library and rename RPM to RPM_input
    # --------------------------------------------------
    input_library = (
        dict_library[tissue][["AA_sequence", "RPM"]]
        .rename(columns={"RPM": "RPM_input"})
        .copy()
    )

    # --------------------------------------------------
    # Select biological replicate RPM values for the chosen
    # tissue and extraction condition
    # --------------------------------------------------
    df = table[
        (table["Tissue"] == tissue) &
        (table["Replicate"] == "biological") &
        (table["Extraction_type"] == extraction)
    ][["AA_sequence", "Mouse_ID", "Extraction_type", "RPM"]].copy()

    # --------------------------------------------------
    # Pivot table: one row per AA_sequence, one column per Mouse_ID
    # --------------------------------------------------
    df_wide = (
        df
        .pivot_table(
            index="AA_sequence",
            columns="Mouse_ID",
            values="RPM",
            aggfunc="first"
        )
        .reset_index()
    )
    df_wide.columns.name = None

    # --------------------------------------------------
    # Get pseudo information from pooled table
    # --------------------------------------------------
    df_pooled_pseudo = df_pooled[
        (df_pooled["Tissue"] == tissue) &
        (df_pooled["Extraction_type"] == extraction)
    ][["AA_sequence", "n_samples_present"]].copy()

    # --------------------------------------------------
    # Merge pseudo information into wide table
    # --------------------------------------------------
    df_wide = df_wide.merge(df_pooled_pseudo, on="AA_sequence", how="inner")

    # --------------------------------------------------
    # Merge input library RPM values
    # --------------------------------------------------
    df_merged = df_wide.merge(
        input_library,
        on="AA_sequence",
        how="inner"
    )

    # --------------------------------------------------
    # Rank input library RPM values
    # Higher RPM receives a lower rank number
    # --------------------------------------------------
    df_merged["Rank_input"] = df_merged["RPM_input"].rank(
        method="average",
        ascending=False
    )

    # --------------------------------------------------
    # Rank each biological replicate separately
    # --------------------------------------------------
    for mouse in MOUSE_ID:
        df_merged[f"Rank_{mouse}"] = df_merged[mouse].rank(
            method="average",
            ascending=False
        )

    # --------------------------------------------------
    # Calculate mean rank across biological replicates
    # --------------------------------------------------
    rank_cols = [f"Rank_{mouse}" for mouse in MOUSE_ID]
    df_merged[f"Rank_mean_{extraction}"] = df_merged[rank_cols].mean(axis=1)

    # --------------------------------------------------
    # Keep only columns relevant for rank-shift plotting
    # --------------------------------------------------
    df_merged = df_merged[
        ["AA_sequence", "Rank_input", f"Rank_mean_{extraction}", "n_samples_present"]
    ].copy()

    # --------------------------------------------------
    # Create figure
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 5))

    # --------------------------------------------------
    # Plot variants within pseudo threshold
    # --------------------------------------------------
    normal = df_merged[df_merged["n_samples_present"] >= n_sample].copy()
    normal_count = normal.shape[0]

    ax.scatter(
        normal["Rank_input"],
        normal[f"Rank_mean_{extraction}"],
        s=5,
        alpha=0.4,
        color="#4C72B0",
        linewidths=0,
        label="Variant"
    )

    # --------------------------------------------------
    # Plot pseudo-supported variants separately
    # --------------------------------------------------
    pseudo = pd.DataFrame()

    if show_pseudo:
        pseudo = df_merged[df_merged["n_samples_present"] < n_sample].copy()
        pseudo_count = pseudo.shape[0]

        ax.scatter(
            pseudo["Rank_input"],
            pseudo[f"Rank_mean_{extraction}"],
            s=5,
            alpha=0.2,
            color="red",
            linewidths=0,
            label=f"n_samples < {n_sample}"
        )

    # --------------------------------------------------
    # Log scaling for rank comparison
    # --------------------------------------------------
    ax.set_xscale("log")
    ax.set_yscale("log")

    # --------------------------------------------------
    # Add diagonal line indicating no rank shift
    # --------------------------------------------------
    max_rank = int(
        max(
            df_merged["Rank_input"].max(),
            df_merged[f"Rank_mean_{extraction}"].max()
        )
    )

    ax.plot(
        [1, max_rank],
        [1, max_rank],
        ls="--",
        lw=1.2,
        c="black",
        alpha=0.4,
        label="no-shift (y=x)"
    )

    # --------------------------------------------------
    # Labels and optional title
    # --------------------------------------------------
    if title:
        ax.set_title(
            f"Rank shift: input vs mean {extraction} {tissue}",
            fontweight="bold"
        )

    ax.set_xlabel("Rank in input library", fontweight="bold")
    ax.set_ylabel(f"Mean rank in {extraction} {tissue}", fontweight="bold")

    ax.legend(
        loc="lower right",
        frameon=True,
        framealpha=0.75,
        edgecolor="0.7"
    )

    plt.tight_layout()

    # --------------------------------------------------
    # Optional saving
    # --------------------------------------------------
    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")

        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            if show_pseudo:
                file_name = f"Rank_shift_input_vs_{extraction}_{tissue}_filter_{n_sample}.png"
            else:
                file_name = f"Rank_shift_input_vs_{extraction}_{tissue}_filter_{n_sample}_without_pseudo.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    # --------------------------------------------------
    # Print summary
    # --------------------------------------------------
    if show_pseudo:
        print(
            f"Variants with Pseudo <= {n_sample}: {normal_count}\n"
            f"Variants with Pseudo > {n_sample}: {pseudo_count}"
        )
    else:
        print(f"Variants with Pseudo <= {n_sample}: {normal_count}")

    plt.show()

    return normal, pseudo, df_merged

## 4. Load files and extract information

### 4.1. Load csv files

In [ ]:
%%time
#load long table
df_long = read_csv_file(tables_dir / 'summary', "df_long_combined_biological_technical_rep")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_sample_pooled")

In [ ]:
print ('Long Table')
display (df_long)

print ('Pooled Table')
display (df_pooled)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
display (SAMPLE, EXT, TISSUE, SEX, MOUSE_ID)

In [ ]:
# Sort different file names in extraction and biological or technical Replicates
DICT_NAMES = sort_file_names (SAMPLE)
DICT_NAMES

### 4.3. Load tissue/ext specific librarys¶

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue in TISSUE:
    df = read_csv_file(tables_dir/tissue, f'df_{tissue}_input_library')
    dict_library[tissue] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

## 5. Figures

### 5.1. Histogram with am RPM with Hue = Sample

In [ ]:
%%time
distribution_histogram(
    table=df_long,
    tissue='heart',
    extraction='gDNA',
    replicate="biological",
    legend=True,
    title=False,
    figsize=(10,5),
    input_alpha= 0.9,
    sample_alpha= 0.4,
    save=True,
    output_path=figures_dir/'1_RPM_histogram_samples_and_input'
)

### 5.2. ECDF RPM with samples and input

In [ ]:
for ext, tissue in product (EXT, TISSUE):
    distribution_ecdf(
        table=df_long,
        tissue=tissue,
        extraction=ext,
        column="RPM",
        replicate="biological",
        no_pseudo=False,
        save = True,
        output_path = figures_dir/'2_ECDF_RPM'     
    )

### 5.3.. Venn2 Input vs mean proportion in sample

In [ ]:
for ext, tis in product (EXT, TISSUE):
    venn2_ntop_AA(
        df_pooled, 
        tissue=tis, 
        extraction=ext, 
        n_top=10000,
        figsize=(7,4),
        title=False, 
        save = True,
        output_path =  figures_dir/'3_Venn2_top_RPM')

### 5.4. Rank shift proportion input library -> gDNA and cDNA

In [ ]:
for tis, ext in product (TISSUE, EXT):

    rank_shift_plot (
        table = df_long, 
        tissue = tis, 
        extraction = ext,
        n_sample = 6, 
        show_pseudo=True,
        title = False, 
        save = True, 
        output_path = figures_dir/'4_Rank_shift'/'mark_all_pseudo', 
        file_name = None)